# Construir el conjunto de referencia de Seminario 1

Este notebook transforma el export **JSON** de Label Studio en el conjunto de referencia local de 400 noticias: `reference_claims.csv`, `train.csv`, `validation.csv` y `test.csv`. No entrena modelos.

Antes de congelar el manifiesto, revisa los pares en `labelstudio_duplicate_review.csv`. Los datos se escriben bajo `artifacts/`, directorio ignorado por Git.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, *cwd.parents) if (path / 'scripts' / 'build_sem1_manifest.py').exists())

# Cambia solo estas rutas si descargas otro export o quieres crear otra corrida.
EXPORT_JSON = Path('/Users/renzo/Downloads/project-9-at-2026-10-31-22-13-fbc549a6.json')
RUN_DIR = ROOT / 'artifacts' / 'sem1_20261031'
FINAL_DIR = RUN_DIR / 'final'
EXCLUSIONS = RUN_DIR / 'reviewed_near_duplicate_exclusions.csv'

assert EXPORT_JSON.exists(), f'No encuentro el export: {EXPORT_JSON}'
assert EXCLUSIONS.exists(), f'Falta revisar/crear: {EXCLUSIONS}'
print({'repo': str(ROOT), 'python': sys.executable, 'export': str(EXPORT_JSON), 'final_dir': str(FINAL_DIR)})

In [ ]:
def run_script(script: str, *arguments: str) -> None:
    command = [sys.executable, script, *map(str, arguments)]
    print(' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)

AUDIT = RUN_DIR / 'labelstudio_claims_audit.csv'
DUPLICATES = RUN_DIR / 'labelstudio_duplicate_review.csv'
AUDIT_SUMMARY = RUN_DIR / 'labelstudio_audit_summary.json'
MANIFEST = FINAL_DIR / 'manifest_final_400.csv'
REFERENCE_SET = FINAL_DIR / 'reference_claims.csv'
TRAIN = FINAL_DIR / 'train.csv'
VALIDATION = FINAL_DIR / 'validation.csv'
TEST = FINAL_DIR / 'test.csv'

## 1. Auditar las 700 tareas

El auditor conserva todas las tareas, detecta claims repetidos, marca similitudes para revisión y rechaza claims que parecen contener el artículo completo.

In [ ]:
run_script(
    'scripts/audit_labelstudio_export.py',
    '--input', EXPORT_JSON,
    '--audit-output', AUDIT,
    '--duplicates-output', DUPLICATES,
    '--summary-output', AUDIT_SUMMARY,
)

summary = json.loads(AUDIT_SUMMARY.read_text(encoding='utf-8'))
summary

In [ ]:
duplicates = pd.read_csv(DUPLICATES)
display(duplicates)

# Este archivo documenta las decisiones humanas sobre pares similares.
# Debe tener, como mínimo, la columna record_id.
display(pd.read_csv(EXCLUSIONS))

## 2. Congelar 400 ejemplos y crear los splits

El manifiesto usa una semilla fija y la proporción 80/10/10: 320 train, 40 validation y 40 test. Si ya existe un manifiesto, no se sobrescribe; para una nueva versión cambia `RUN_DIR`.

In [ ]:
if MANIFEST.exists():
    print(f'El conjunto de referencia ya está congelado en: {FINAL_DIR}')
else:
    run_script(
        'scripts/build_sem1_manifest.py',
        '--input', EXPORT_JSON,
        '--output', MANIFEST,
        '--exclude-record-ids', EXCLUSIONS,
    )
    run_script(
        'scripts/import_reference_set.py',
        '--input', EXPORT_JSON,
        '--manifest', MANIFEST,
        '--output', REFERENCE_SET,
        '--require-complete',
    )
    run_script(
        'scripts/build_sem1_splits.py',
        '--input', REFERENCE_SET,
        '--train-output', TRAIN,
        '--validation-output', VALIDATION,
        '--test-output', TEST,
    )

## 3. Verificación final

Estos son los archivos que usarás después: `train.csv` para entrenar, `validation.csv` para elegir checkpoint y `test.csv` únicamente para las métricas finales.

In [ ]:
references = pd.read_csv(REFERENCE_SET)
claim_words = references['reference_claim'].fillna('').str.split().str.len()
checks = {
    'rows': len(references),
    'splits': references['split'].value_counts().sort_index().to_dict(),
    'empty_claims': int(references['reference_claim'].fillna('').str.strip().eq('').sum()),
    'unique_sources': int(references['source_hash'].nunique()),
    'unique_claims': int(references['claim_hash'].nunique()),
    'min_claim_words': int(claim_words.min()),
    'max_claim_words': int(claim_words.max()),
    'files': [str(path) for path in (MANIFEST, REFERENCE_SET, TRAIN, VALIDATION, TEST)],
}
assert checks['rows'] == 400
assert checks['splits'] == {'test': 40, 'train': 320, 'validation': 40}
assert checks['empty_claims'] == 0
assert checks['unique_sources'] == 400
assert checks['unique_claims'] == 400
assert checks['min_claim_words'] >= 5
assert checks['max_claim_words'] <= 80
checks